In [1]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

llm = init_chat_model("openai:gpt-5-nano")

conn = sqlite3.connect("memory.db", check_same_thread=False)

In [2]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [3]:
@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])  # langchain api


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])  # langchain api
    return {"messages": [response]}

In [4]:
tool_node = ToolNode(
    tools=[get_weather],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

In [ ]:
graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)

graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather in Seoul?"},
        ]
    },
    config={
        "configurable": {
            "thread_id": "1",
        },
        # "recursion_limit": 2,  # graph step 제한 (ex.fail safe, infinite loop)
    },
)

In [ ]:
for state in graph.get_state_history(
    {
        "configurable": {
            "thread_id": "1",
        }
    }
):
    print(state.next)  # bottom -> top

()
('chatbot',)
('__start__',)
('chatbot',)
('tools',)
('chatbot',)
('__start__',)


In [ ]:
# stream
graph = graph_builder.compile()

async for event in graph.astream(
    {
        "messages": [
            {"role": "user", "content": "What is the weather in Seoul?"},
        ]
    },
    stream_mode="messages",
):
    print(event)

(AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_ydtI9G2PeTBzChtjcfLRCy95', 'function': {'arguments': '', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={}, id='run--019d293a-8c92-75e0-be66-239559f5ab94', tool_calls=[{'name': 'get_weather', 'args': {}, 'id': 'call_ydtI9G2PeTBzChtjcfLRCy95', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_weather', 'args': '', 'id': 'call_ydtI9G2PeTBzChtjcfLRCy95', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 1, 'langgraph_node': 'chatbot', 'langgraph_triggers': ('branch:to:chatbot',), 'langgraph_path': ('__pregel_pull', 'chatbot'), 'langgraph_checkpoint_ns': 'chatbot:250175ba-7e5f-4e73-64f6-f656afe42ff4', 'checkpoint_ns': 'chatbot:250175ba-7e5f-4e73-64f6-f656afe42ff4', 'ls_provider': 'openai', 'ls_model_name': 'gpt-5-nano', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': None, 'function': {'ar

In [ ]:
"""
graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather in Seoul?"},
        ]
    },
    config={
        "configurable": {
            "thread_id": "1",
        },
        "recursion_limit": 2,  # graph step 제한 (ex.fail safe, infinite loop)
    },
)
"""

# GraphRecursionError: Recursion limit of 2 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.